# FFTW C2C Cython Sequential

In [41]:
#=-----------------------------------------------------------------------=#

In [3]:
! cython --version

Cython version 0.29.21


In [10]:
%%writefile cc2cs.pyx
#cython: boundscheck=False, wraparound=False, cdivision=True
#cython: initializedcheck=False, language_level=3, infer_types=True
import numpy as np, pyfftw as pf, time as tm
def ff():
    t0 = tm.time()    # <--- time measurement
    N = 500
    a = pf.empty_aligned( (N, N, N), dtype=np.complex128 )
    a[:,:,:] = np.fromfunction( lambda i, j, k :
        (i*N**2 + j*N  + k + 1)*1E-6,
        (N, N, N), dtype=np.complex128 )
    b = pf.interfaces.numpy_fft.fftn(a)
    s = np.sum(b)
    t1 = tm.time()    # <--- time measurement
    return s, t1-t0

Overwriting cc2cs.pyx


In [11]:
! python cc2cs.pyx

In [12]:
%%writefile setup.py
from setuptools import setup
from Cython.Build import cythonize

setup(
    ext_modules = cythonize("cc2cs.pyx")
)

Overwriting setup.py


In [ ]:
%%bash
#module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
#source /scratch/app/anaconda3/2020.11/etc/profile.d/conda.sh
#conda activate /scratch/app/anaconda3/2020.11
#conda activate --stack ../envs
#rm cc2cs*.so
#LDFLAGS="-L $(pwd)/lo/lib -l fftw3 -l m"  \
#CFLAGS="-I $(pwd)/lo/include"  \
python setup.py build_ext --inplace

In [14]:
! ls cc2cs*.so

cc2cs.cpython-39-x86_64-linux-gnu.so


In [1]:
import cc2cs
print(cc2cs.__doc__)

None


In [2]:
import cc2cs
cc2cs.ff()
s,t=cc2cs.ff()
print('S: {:.2f}    T: {:.4f} s'.format(s, t))

S: 125.00-0.00j    T: 24.2911 s


In [3]:
%%writefile cc2cs_c.py
import cc2cs
cc2cs.ff()
s,t=cc2cs.ff()
print('S: {:.2f}    T: {:.4f} s'.format(s, t))

Writing cc2cs_c.py


In [4]:
! python cc2cs_c.py

S: 125.00-0.00j    T: 19.1375 s


In [5]:
! cp cc2cs_c.py cc2cs*.so /scratch${PWD#"/prj"}

In [16]:
%%writefile cc2cs.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu_dev  : 20 min.,  1-4  nodes, 1/1   tasks
#   cpu_small: 72 hours, 1-20 nodes, 16/96 tasks
#SBATCH --partition cpu_small  # Select partition
#SBATCH --ntasks=1             # Total tasks
#SBATCH --job-name cc2cs       # Job name
#SBATCH --time=00:01:00        # Limit execution time
#SBATCH --exclusive            # Exclusive acccess to nodes

echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST

# Environment
cd
cd /scratch${PWD#"/prj"}/    # maybe wrong, need check
module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
source /scratch/app/anaconda3/2020.11/etc/profile.d/conda.sh
conda activate --stack ./envs
cd fft
                                              
# Executable
EXEC="python cc2cs_c.py"

# Start
echo '$ srun -n' $SLURM_NTASKS ${EXEC##*/}
echo '-- output -----------------------------'
srun -n $SLURM_NTASKS $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting cc2cs.srm


In [7]:
! sbatch --partition=cpu_dev cc2cs.srm

Submitted batch job 869439


In [10]:
! squeue -n pc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            869439    cpu_dev   R  0:18     1   24


In [14]:
! squeue -n pc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [15]:
! cat /scratch${PWD#"/prj"}/slurm-869439.out

- Job ID: 869439
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1257
$ srun -n 1 python cc2cs_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 19.9773 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [17]:
! sbatch cc2cs.srm
! sbatch cc2cs.srm

Submitted batch job 869440
Submitted batch job 869441


In [18]:
! squeue -n cc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            869440  cpu_small  PD  0:00     1    1
            869441  cpu_small  PD  0:00     1    1


In [1]:
! squeue -n cc2cs -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-869440.out
! cat /scratch${PWD#"/prj"}/slurm-869441.out

- Job ID: 869440
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
$ srun -n 1 python cc2cs_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 20.3121 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 869441
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1317
$ srun -n 1 python cc2cs_c.py
-- output -----------------------------
S: 125.00-0.00j    T: 16.7662 s
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
